# 📗 부록 — 데코레이터(`@`) 기초

> **이 노트북은 수업 시간에 다루지 않는 참고 자료입니다.** 완주 기준에 들어가지 않습니다. `@tool` 의 `@` 가 눈에 걸릴 때 펴 보세요.

교안에서 도구를 만들 때 함수 위에 이렇게 한 줄을 붙였습니다.

```python
@tool
def supply_stock(item_name: str) -> int:
    """비품 이름을 받아 현재 창고 재고 수량을 돌려준다."""
    return _STOCK.get(item_name.strip(), 0)
```

이 **`@tool`** 한 줄이 무엇인지 설명 없이 지나왔습니다. 이것을 **데코레이터**라고 부릅니다. 이 부록은 데코레이터가 무엇인지 **처음부터** 풀어 봅니다. 다 읽고 나면 위 코드가 **`supply_stock = tool(supply_stock)`** 을 줄여 쓴 것임을 알게 됩니다.

> **모델을 한 번도 부르지 않습니다.** 순수 파이썬이라 API 키 없이 처음부터 끝까지 돌아가고, 출력도 항상 같습니다.

**이 부록에서 하는 것**

- [ ] 파이썬에서 **함수도 값**이라는 것 — 변수에 담고, 인자로 넘긴다
- [ ] **함수를 돌려주는 함수**를 만들어 다른 함수를 감싼다
- [ ] **`@`** 가 그 감싸기를 줄여 쓰는 표기임을 안다
- [ ] **아무 함수나** 감쌀 수 있게 만든다(`*args`·`**kwargs`)
- [ ] 감싸면 **원래 이름이 사라지는** 문제와 `functools.wraps` 로 고치는 법
- [ ] 그 눈으로 **`@tool`** 을 다시 읽는다

---
# 1. 함수도 값이다

데코레이터를 이해하려면 먼저 이것부터 받아들여야 합니다 — **파이썬에서 함수는 값입니다.** 숫자나 문자열처럼 **변수에 담을 수 있고, 다른 함수에 넘길 수도 있습니다.**

괄호를 붙이면 **부르는 것**이고, 괄호를 빼면 **함수 그 자체**를 가리킵니다. 이 차이가 전부입니다.

In [ ]:
def greet(name):
    return f'안녕하세요, {name}님'

# 괄호를 붙이면 '부른다' -> 결과 문자열이 나온다
print(greet('민수'))

# 괄호를 빼면 '함수 자체' -> 함수 객체가 나온다
print(greet)

괄호 없이 쓴 `greet` 는 **함수 객체**입니다. 값이니까 다른 변수에 담을 수 있습니다.

In [ ]:
# 함수를 변수에 담는다 - 복사가 아니라 '같은 함수에 이름을 하나 더 붙이는' 것이다
hello = greet

print(hello('민수'))          # 새 이름으로도 똑같이 불린다
print(hello is greet)         # 같은 함수인가?

값이니까 **함수에 인자로 넘길** 수도 있습니다. 아래 `run_twice` 는 **함수를 받아** 두 번 부릅니다.

In [ ]:
def run_twice(func, value):
    """함수와 값을 받아, 그 함수를 값에 두 번 적용한다."""
    first = func(value)
    second = func(value)
    return [first, second]


# greet 를 '부르지 않고' 넘긴다 - 괄호가 없다는 데 주목
print(run_twice(greet, '민수'))

# 파이썬이 원래 갖고 있는 함수도 마찬가지다
print(run_twice(len, '데코레이터'))

> **여기까지가 데코레이터의 전제입니다.** 함수를 값처럼 주고받을 수 있으니, **함수를 받아서 새 함수를 돌려주는** 함수도 만들 수 있습니다. 그게 데코레이터입니다.

---
# 2. 함수를 돌려주는 함수 — 감싸기

이제 한 걸음 더 갑니다. **함수를 받아서, 그 함수를 감싼 새 함수를 돌려주는** 함수를 만들어 봅니다.

아래 `shout` 는 `greet` 를 받아 **`greet` 를 부른 뒤 그 결과에 느낌표를 붙이는** 새 함수를 만들어 돌려줍니다. 원래 함수는 **건드리지 않습니다** — 겉을 한 겹 두르는 것뿐입니다.

In [ ]:
def shout(func):
    """함수를 받아, 그 결과를 대문자처럼 강조해 돌려주는 새 함수를 만든다."""

    # 안쪽에서 새 함수를 정의한다 - 이 함수가 '감싼 것' 이다
    def wrapper(name):
        result = func(name)        # 원래 함수를 부르고
        return result + '!!!'      # 그 결과에 한 겹 얹는다

    return wrapper                 # 부르지 않고 '함수 자체' 를 돌려준다


# shout 에 greet 를 넘기면 새 함수가 나온다
loud_greet = shout(greet)

print(greet('민수'))         # 원래 함수는 그대로다
print(loud_greet('민수'))    # 감싼 함수는 느낌표가 붙는다

핵심은 두 가지입니다.

1. `shout` 안에서 **새 함수 `wrapper` 를 정의**하고, `return wrapper` 로 **그 함수를 돌려줍니다**(`wrapper()` 가 아니라 `wrapper` — 괄호가 없습니다).
2. `wrapper` 안에서 **원래 함수 `func` 를 부릅니다.** 그래서 원래 하던 일은 그대로 하면서 앞뒤로 무언가를 더할 수 있습니다.

이제 이름을 바꿔치기해 봅니다. 감싼 결과를 **원래 이름에 다시 담으면**, 겉보기에는 `greet` 가 업그레이드된 것처럼 보입니다.

In [ ]:
def greet(name):
    return f'안녕하세요, {name}님'


# 감싼 결과를 '같은 이름' 에 다시 담는다
greet = shout(greet)

print(greet('민수'))   # greet 를 불렀는데 느낌표가 붙어 나온다

> **이 한 줄이 데코레이터의 전부입니다.**
>
> ```python
> greet = shout(greet)
> ```
>
> 함수를 감싸는 함수에 넣고, 그 결과를 **같은 이름에 다시 담는 것.** 다음 절에서 볼 `@` 는 바로 이 줄을 줄여 쓰는 표기일 뿐입니다.

---
# 3. `@` 는 줄여 쓰는 표기다

앞 절의 `greet = shout(greet)` 는 문제가 하나 있습니다 — **함수 정의와 감싸는 줄이 떨어져 있습니다.** 함수가 길면 아래쪽 그 한 줄을 놓치기 쉽고, 읽는 사람도 이 함수가 감싸졌는지 끝까지 내려가 봐야 압니다.

그래서 파이썬은 **함수 정의 바로 위에 `@감쌀함수` 를 적으면 같은 일을 하도록** 표기를 하나 만들어 두었습니다. 두 코드는 **완전히 같습니다.**

| 손으로 쓰면 | `@` 로 쓰면 |
|---|---|
| `def greet(name): ...`<br>`greet = shout(greet)` | `@shout`<br>`def greet(name): ...` |

In [ ]:
@shout
def welcome(name):
    return f'환영합니다, {name}님'


# 위 두 줄은 아래와 완전히 같다:
#     def welcome(name): ...
#     welcome = shout(welcome)
print(welcome('민수'))

정말 같은지 **직접 확인**해 봅시다. `@` 없이 손으로 감싼 것과 `@` 로 감싼 것의 결과를 나란히 찍어 비교합니다.

In [ ]:
# (가) @ 없이 손으로 감싸기
def bye_manual(name):
    return f'안녕히 가세요, {name}님'

bye_manual = shout(bye_manual)


# (나) @ 로 감싸기
@shout
def bye_deco(name):
    return f'안녕히 가세요, {name}님'


print('손으로:', bye_manual('민수'))
print('@ 로  :', bye_deco('민수'))
print('결과가 같은가?', bye_manual('민수') == bye_deco('민수'))

### 🖐️ 함께 따라하기 — 괄호로 감싸는 데코레이터

결과 문자열을 **대괄호로 감싸는** 데코레이터를 직접 만들어 보세요.

1. `bracket(func)` 함수를 만듭니다. 안에서 `wrapper(text)` 를 정의하고, `func(text)` 결과를 `'[' + 결과 + ']'` 로 만들어 돌려준 뒤, `wrapper` 를 반환합니다.
2. `@bracket` 을 붙인 `title(text)` 함수를 만듭니다 — 받은 글자를 그대로 돌려주면 됩니다.
3. `title('공지사항')` 을 출력합니다.

**확인 기준**: `[공지사항]` 이 찍힌다.

In [ ]:
# 여기에 코드를 작성하세요
# 1) bracket(func) 를 만든다 (안에서 wrapper(text) 정의 -> func(text) 결과를 대괄호로 감싸 반환 -> return wrapper)
# 2) @bracket 을 붙인 title(text) 를 만든다 (받은 글자를 그대로 반환)
# 3) title('공지사항') 을 출력한다

---
# 4. 아무 함수나 감싸려면

지금까지 만든 `shout` 에는 약점이 있습니다. `wrapper(name)` 이 **인자를 딱 하나만** 받도록 적혀 있어서, **인자가 두 개인 함수는 감쌀 수 없습니다.** 직접 부딪혀 봅시다.

In [ ]:
@shout
def add(a, b):
    return f'{a} + {b} = {a + b}'


# wrapper 는 인자를 하나만 받게 만들어져 있어서 두 개를 넣으면 터진다
try:
    print(add(2, 3))
except TypeError as e:
    print('TypeError:', e)

감싸는 쪽은 원래 함수가 **인자를 몇 개 받는지 미리 알 수 없습니다.** 그래서 **"몇 개가 오든 그대로 받아서 그대로 넘긴다"** 라고 적는 표기가 필요합니다. 그것이 **`*args`** 와 **`**kwargs`** 입니다.

| 표기 | 받는 것 | 모양 |
|---|---|---|
| `*args` | 이름 없이 순서대로 넘긴 인자들 | 튜플 — `add(2, 3)` 이면 `(2, 3)` |
| `**kwargs` | `이름=값` 으로 넘긴 인자들 | 딕셔너리 — `add(a=2, b=3)` 이면 `{'a': 2, 'b': 3}` |

먼저 이 둘이 무엇을 담는지 눈으로 봅니다.

In [ ]:
def show(*args, **kwargs):
    """받은 인자가 어떻게 담기는지 그대로 보여 준다."""
    print('args  =', args)
    print('kwargs =', kwargs)


show(2, 3)
print('-' * 30)
show(a=2, b=3)
print('-' * 30)
show(2, b=3)

이제 `wrapper` 를 `*args, **kwargs` 로 고치면 **인자가 몇 개든** 감쌀 수 있습니다. 받을 때 별을 붙여 **모으고**, 넘길 때 별을 붙여 **다시 펼칩니다.**

In [ ]:
def shout_any(func):
    """인자 개수와 상관없이 아무 함수나 감싼다."""

    def wrapper(*args, **kwargs):      # 몇 개가 오든 모아서 받고
        result = func(*args, **kwargs)  # 별을 붙여 그대로 펼쳐 넘긴다
        return result + '!!!'

    return wrapper


@shout_any
def add2(a, b):
    return f'{a} + {b} = {a + b}'


@shout_any
def hi(name):
    return f'안녕, {name}'


print(add2(2, 3))        # 인자 두 개 - 이제 된다
print(add2(a=2, b=3))    # 이름을 붙여 넘겨도 된다
print(hi('민수'))         # 인자 한 개도 그대로 된다

---
# 5. 감싸면 원래 이름이 사라집니다

데코레이터에는 잘 알려진 부작용이 하나 있습니다. 감싸고 나면 **함수의 이름과 설명이 `wrapper` 의 것으로 바뀝니다.** `greet = shout(greet)` 를 떠올려 보면 당연합니다 — 그 이름에 담긴 것은 이제 `wrapper` 니까요.

In [ ]:
@shout_any
def introduce(name):
    """이름을 받아 자기소개 문장을 만든다."""
    return f'저는 {name}입니다'


# 우리가 적은 이름·설명이 아니라 wrapper 의 것이 나온다
print('이름:', introduce.__name__)
print('설명:', introduce.__doc__)

**교재에서 이것이 왜 중요한가요?** 교안에서 `@tool` 을 배울 때 **"함수 이름이 도구 이름이 되고, docstring 이 도구 설명이 된다"** 고 했습니다. 만약 감싸면서 이름과 설명이 사라진다면 도구가 전부 `wrapper` 라는 이름을 갖게 되겠지요.

그래서 파이썬 표준 라이브러리는 **원래 함수의 이름·설명을 `wrapper` 에 옮겨 붙여 주는** 도구를 제공합니다 — `functools.wraps` 입니다. `wrapper` 정의 위에 한 줄 붙이면 끝입니다(이것 자체도 데코레이터입니다).

In [ ]:
import functools


def shout_keep(func):
    """원래 함수의 이름·설명을 유지하면서 감싼다."""

    @functools.wraps(func)          # <- 이 한 줄이 이름·설명을 옮겨 붙인다
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + '!!!'

    return wrapper


@shout_keep
def introduce2(name):
    """이름을 받아 자기소개 문장을 만든다."""
    return f'저는 {name}입니다'


print('이름:', introduce2.__name__)   # 이제 우리가 적은 이름이 그대로다
print('설명:', introduce2.__doc__)
print('동작:', introduce2('민수'))     # 감싼 기능도 그대로다

### 🖐️ 함께 따라하기 — 부를 때마다 기록을 남기는 데코레이터

실무에서 가장 흔한 쓰임 하나를 만들어 봅니다 — **함수가 불릴 때마다 무엇으로 불렸는지 찍어 주는** 데코레이터입니다.

1. `log_call(func)` 를 만듭니다.
   - 안쪽 `wrapper(*args, **kwargs)` 위에 `@functools.wraps(func)` 를 붙입니다.
   - `wrapper` 안에서 먼저 `f'[호출] {func.__name__} args={args}'` 를 출력합니다.
   - 그다음 `func(*args, **kwargs)` 를 불러 결과를 받고, `f'[반환] {결과}'` 를 출력한 뒤 그 결과를 돌려줍니다.
2. `@log_call` 을 붙인 `multiply(a, b)` 를 만듭니다 — `a * b` 를 돌려줍니다.
3. `multiply(3, 4)` 를 부르고, 마지막에 `multiply.__name__` 도 출력합니다.

**확인 기준**: `[호출] multiply args=(3, 4)` · `[반환] 12` 가 차례로 찍히고, `multiply.__name__` 이 `wrapper` 가 아니라 **`multiply`** 로 나온다.

In [ ]:
# 여기에 코드를 작성하세요
# 1) log_call(func) 를 만든다 (@functools.wraps(func) 를 붙인 wrapper(*args, **kwargs))
#    - 부르기 전: f'[호출] {func.__name__} args={args}' 출력
#    - func(*args, **kwargs) 결과를 받아 f'[반환] {결과}' 출력 후 그 결과를 반환
# 2) @log_call 을 붙인 multiply(a, b) 를 만든다 (a * b 반환)
# 3) multiply(3, 4) 를 부르고 multiply.__name__ 도 출력한다

---
# 6. 이제 `@tool` 을 다시 읽습니다

여기까지 왔으면 교안의 그 한 줄이 무엇이었는지 읽힙니다.

```python
@tool
def supply_stock(item_name: str) -> int:
    """비품 이름을 받아 현재 창고 재고 수량을 돌려준다."""
    return _STOCK.get(item_name.strip(), 0)
```

이것은 **`supply_stock = tool(supply_stock)`** 입니다. 우리가 만든 `shout` 가 결과에 느낌표를 붙인 것처럼, LangChain 의 `tool` 은 우리 함수를 받아 **모델이 쓸 수 있는 도구 객체**로 바꿔 돌려줍니다.

그래서 교안에서 본 두 가지가 자연스럽게 설명됩니다.

- **왜 함수 이름이 도구 이름이 되고 docstring 이 도구 설명이 되는가** — `tool` 이 감싸면서 원래 함수의 `__name__`·`__doc__` 을 읽어 가기 때문입니다(5절에서 본 그 이름·설명입니다).
- **왜 `@tool` 을 붙여도 그냥 함수처럼 쓸 수 있는가** — 감싼 결과가 원래 하던 일을 그대로 하기 때문입니다. 교안에서 도구를 모델 없이 직접 불러 본 것이 그 확인이었습니다.

차이는 하나뿐입니다. 우리 `shout` 는 **함수**를 돌려줬지만, `tool` 은 이름·설명·인자 명세를 함께 들고 있는 **도구 객체**를 돌려줍니다. 그래서 `.name`·`.description` 처럼 점으로 꺼내 볼 수 있었던 것입니다.

### ✅ 바로 확인 퀴즈

**1.** `@shout` 를 붙인 것과 같은 일을 `@` 없이 쓰면 어떻게 되나요?

<details><summary>정답 보기</summary>

함수를 정의한 뒤 **`이름 = shout(이름)`** 으로 감싼 결과를 같은 이름에 다시 담으면 됩니다. `@` 는 이 줄을 함수 정의 바로 위로 옮겨 적는 표기일 뿐입니다.

</details>

**2.** `wrapper` 를 `*args, **kwargs` 로 적는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

감싸는 쪽은 원래 함수가 **인자를 몇 개 받는지 알 수 없기** 때문입니다. `*args, **kwargs` 로 **몇 개가 오든 모아서 받고**, 넘길 때 별을 붙여 **그대로 펼쳐** 전달하면 인자 개수와 상관없이 감쌀 수 있습니다.

</details>

**3.** `functools.wraps` 를 빼면 `@tool` 로 만든 도구에 어떤 문제가 생길까요?

<details><summary>정답 보기</summary>

도구의 **이름과 설명이 원래 함수의 것이 아니게** 됩니다. 도구 이름이 `wrapper` 가 되고 docstring 도 사라지니, 모델이 **"언제 이 도구를 쓰는지"** 를 판단할 근거를 잃습니다. 교안에서 docstring 이 곧 명세라고 한 것이 이것과 이어집니다.

</details>

---
## 정리

| 개념 | 핵심 |
|---|---|
| 함수도 값 | 괄호를 빼면 함수 자체 — 변수에 담고 인자로 넘길 수 있다 |
| 데코레이터 | **함수를 받아 감싼 함수를 돌려주는** 함수 |
| `@감쌀함수` | `이름 = 감쌀함수(이름)` 을 줄여 쓴 표기 |
| `*args`·`**kwargs` | 인자를 **몇 개가 오든** 모아서 받고 그대로 펼쳐 넘긴다 |
| `functools.wraps` | 감쌀 때 **원래 이름·설명을 유지**한다 |
| `@tool` | `함수 = tool(함수)` — 함수를 **도구 객체**로 바꿔 준다 |

> 더 깊은 것들(인자를 받는 데코레이터, 클래스 데코레이터, 여러 개 쌓기)은 이 부록에서 일부러 다루지 않았습니다. **`@tool` 을 읽는 데는 여기까지면 충분합니다.**